# Practical 10: Credit Card Fraud Detection using Deep Learning

**Problem Statement:** Detect fraudulent credit card transactions using deep neural networks.

**Activities:**
1. Handle imbalanced data
2. Train classification model
3. Evaluate precision, recall, and F1-score

**Dataset:** Credit Card Fraud Detection dataset (284,807 transactions, 492 fraudulent), loaded via OpenML.

## 1. Import Libraries and Load Dataset

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight

data = fetch_openml(data_id=1597, as_frame=True, parser='auto')
df = data.frame
df['Class'] = df['Class'].astype(int)
df.head()

## 2. Explore Class Imbalance

In [ ]:
class_counts = df['Class'].value_counts()
print("Class distribution:\n", class_counts)
print(f"\nFraud percentage: {class_counts[1] / len(df) * 100:.4f}%")

class_counts.plot(kind='bar')
plt.title('Class Distribution (0 = Legitimate, 1 = Fraud)')
plt.ylabel('Number of Transactions')
plt.show()

## 3. Data Preprocessing and Handling Imbalance

The V1-V28 features are already PCA-transformed and scaled; only `Time` and `Amount` need standardization. Since fraud cases are a small minority, class weights are computed from the training set and passed to the model so it penalizes mistakes on the minority class more heavily.

In [ ]:
X = df.drop(columns=['Class'])
y = df['Class']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train[['Time', 'Amount']] = scaler.fit_transform(X_train[['Time', 'Amount']])
X_test[['Time', 'Amount']] = scaler.transform(X_test[['Time', 'Amount']])

class_weights = compute_class_weight('balanced', classes=np.array([0, 1]), y=y_train)
class_weight_dict = {0: class_weights[0], 1: class_weights[1]}

print("Train shape:", X_train.shape, "Test shape:", X_test.shape)
print("Class weights:", class_weight_dict)

## 4. Build and Train Classification Model

In [ ]:
model = keras.Sequential([
    keras.layers.Input(shape=(X_train.shape[1],)),
    keras.layers.Dense(64, activation='relu'),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(32, activation='relu'),
    keras.layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy', keras.metrics.Precision(name='precision'), keras.metrics.Recall(name='recall')]
)

history = model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=20,
    batch_size=2048,
    class_weight=class_weight_dict,
    verbose=1
)

## 5. Evaluate Precision, Recall, and F1-Score

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

y_pred_prob = model.predict(X_test, verbose=0).flatten()
y_pred = (y_pred_prob > 0.5).astype(int)

print(classification_report(y_test, y_pred, target_names=['Legitimate', 'Fraud'], digits=4))

cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Legitimate', 'Fraud'])
disp.plot(cmap='Blues')
plt.title('Confusion Matrix')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history.history['loss'], label='Training Loss')
axes[0].plot(history.history['val_loss'], label='Validation Loss')
axes[0].set_title('Loss over Epochs')
axes[0].set_xlabel('Epoch')
axes[0].legend()

axes[1].plot(history.history['precision'], label='Training Precision')
axes[1].plot(history.history['recall'], label='Training Recall')
axes[1].set_title('Precision and Recall over Epochs')
axes[1].set_xlabel('Epoch')
axes[1].legend()

plt.tight_layout()
plt.show()

## Conclusion

In this practical, we:
- Explored the severe class imbalance in credit card fraud data
- Handled the imbalance using computed class weights
- Trained a deep neural network classifier
- Evaluated performance using precision, recall, F1-score, and a confusion matrix